# Project Patho · Colab slide processing
Run from top to bottom. This notebook loads the saved model for **inference only**. It never calls model.fit, overwrites pilot files, or opens a paid cloud server.

Default: a bounded **technical smoke test** on the public CMU-1-Small-Region SVS. This is a skin reference image, not a lymph-node evaluation set. Its predictions test file/patch/result plumbing and must not be interpreted as diagnostic performance.

The input profile uses the investigator-confirmed original PCam scale, nominally 0.972 microns per pixel (0.243 at 40x, undersampled to 10x; github.com/basveeling/pcam#patch-selection). For a research run, use your own deidentified target-domain slide and set PURPOSE to research. Magnification alone is insufficient without scanner calibration. Free Colab resources can disconnect.

Output: a new timestamped folder in Pathology_Project/Paper_2/Colab_Runs, containing CSV checkpoints, configuration, manifest and dashboard-result.json. Download the JSON and choose **Import model results** at https://projectpatho.pages.dev. The embedded image is an overview; full-resolution zoom still uses the local workstation. Partial coverage and technical-test status are shown explicitly.


In [ ]:
# 1. Connect Drive. Complete Google's authorization prompt yourself if shown.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/Pathology_Project')
assert ROOT.is_dir(), 'Pathology_Project folder was not found.'
MODEL_PATH = ROOT / 'my_cancer_model.h5'
assert MODEL_PATH.is_file(), 'Set MODEL_PATH to the existing saved model. Do not retrain it.'
print('Saved model found:', MODEL_PATH.name)


In [ ]:
# 2. Install only the slide reader; use Colab's TensorFlow environment.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openslide-python==1.4.7', 'openslide-bin==4.0.1.2'])
import tensorflow as tf
import numpy as np
print('TensorFlow:', tf.__version__, 'NumPy:', np.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## Configure a bounded run
The default scans at most 2,000 candidate supports in row-major order. It does not claim complete slide coverage. MAX_CANDIDATES includes background candidates, so some runs may score fewer patches. This tissue filter is a heuristic and can skip pathology. Increase the cap or select a grid-aligned ROI only after checking registration and resource use.

For your own slide, set USE_PUBLIC_SAMPLE=False and SLIDE_PATH to its Drive path. Use the existing local dashboard if native-resolution zoom is required. No slide or model file is published to Cloudflare.


In [ ]:
# 3. Run configuration
USE_PUBLIC_SAMPLE = True
SLIDE_PATH = ROOT / 'Paper_2' / 'Slides' / 'your_slide.svs'  # used only when USE_PUBLIC_SAMPLE=False
PURPOSE = 'technical_smoke_test'  # 'research' requires confirmed target pixel size below
TARGET_MPP = 0.972  # nominal PCam scale: 0.243 um/px at 40x, downsampled 4x to 10x
CONFIRMED_MODEL_PIXEL_SCALE = True  # investigator confirmed the original PCam input scale
MAX_CANDIDATES = 2000
BATCH_SIZE = 16
MIN_TISSUE_FRACTION = 0.10
ROI = None  # optional [x, y, width, height] in analysis pixels; x/y must be multiples of 32

if PURPOSE == 'research':
    assert CONFIRMED_MODEL_PIXEL_SCALE and TARGET_MPP is not None, 'Confirm the model input physical scale before a research run.'
if USE_PUBLIC_SAMPLE:
    import urllib.request
    SLIDE_PATH = Path('/content/CMU-1-Small-Region.svs')
    if not SLIDE_PATH.exists():
        urllib.request.urlretrieve('https://openslide.cs.cmu.edu/download/openslide-testdata/Aperio/CMU-1-Small-Region.svs', SLIDE_PATH)
    assert PURPOSE == 'technical_smoke_test', 'The public skin sample is for technical checks, not model evaluation.'
assert SLIDE_PATH.is_file(), f'Slide not found: {SLIDE_PATH}'
print('Purpose:', PURPOSE, 'Slide:', SLIDE_PATH.name, 'Candidate cap:', MAX_CANDIDATES)


In [ ]:
# 4. Processing engine (self-contained; inference-only)
"""Bounded, inference-only slide processing and browser review export."""
import base64
import csv
import datetime
import hashlib
import io
import json
import math
import platform
import time
from pathlib import Path

import numpy as np
import openslide
from PIL import Image

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def scale_for(slide,target_mpp,purpose):
    if purpose=='technical_smoke_test' and target_mpp is None:
        return 1.0
    if purpose not in ('research','technical_smoke_test') or not isinstance(target_mpp,(int,float)) or not math.isfinite(target_mpp) or target_mpp<=0:
        raise ValueError('Research runs require a confirmed target_mpp in microns per model pixel.')
    try:
        x=float(slide.properties['openslide.mpp-x']);y=float(slide.properties['openslide.mpp-y'])
    except (KeyError,ValueError):
        raise ValueError('Slide has no usable pixel scale; verify its scanner metadata first.')
    if not all(math.isfinite(v) and v>0 for v in (x,y)) or abs(x-y)/x>.01:
        raise ValueError('This version requires verified, approximately square source pixels.')
    scale=target_mpp/x
    if not .25<=scale<=32:
        raise ValueError('Requested scale is outside this implementation’s supported bounds.')
    return scale

def context(slide,x,y,scale):
    """x,y are central-support origin in analysis pixels; pad context white."""
    level=slide.get_best_level_for_downsample(scale)
    native=slide.level_downsamples[level]
    extent=max(1,math.ceil(96*scale/native))
    rgba=slide.read_region((round((x-32)*scale),round((y-32)*scale)),level,(extent,extent))
    background=Image.new('RGBA',rgba.size,'white')
    image=Image.alpha_composite(background,rgba).convert('RGB').resize((96,96),Image.Resampling.BILINEAR)
    return np.asarray(image,dtype=np.uint8)

def tissue_fraction(image):
    center=image[32:64,32:64].astype(np.int16)
    return float(((center.mean(axis=2)<235)&((center.max(axis=2)-center.min(axis=2))>12)).mean())

def process_slide(slide_path,predictor,output_dir,*,model_sha256,model_name,purpose='technical_smoke_test',target_mpp=None,max_candidates=2000,batch_size=16,min_tissue=.1,roi=None,source_slide_id=None):
    if not 1<=max_candidates<=100000 or not 1<=batch_size<=32 or not 0<=min_tissue<=1:
        raise ValueError('Invalid candidate, batch or tissue limit.')
    if not isinstance(model_sha256,str) or len(model_sha256)!=64:
        raise ValueError('Record the actual saved model SHA-256.')
    source=Path(slide_path)
    slide=openslide.OpenSlide(str(source))
    try:
        slide.set_cache(openslide.OpenSlideCache(64*1024*1024))
        scale=scale_for(slide,target_mpp,purpose)
        width,height=slide.dimensions
        aw,ah=math.ceil(width/scale),math.ceil(height/scale)
        cols,rows=math.ceil(aw/32),math.ceil(ah/32)
        if rows*cols>10**10:
            raise ValueError('Slide grid exceeds the viewer contract.')
        if roi is None: roi=[0,0,aw,ah]
        if len(roi)!=4 or not all(isinstance(v,int) for v in roi):
            raise ValueError('ROI is [x,y,width,height] in analysis pixels, using integers.')
        rx,ry,rw,rh=roi
        if rx<0 or ry<0 or rw<=0 or rh<=0 or rx+rw>aw or ry+rh>ah or rx%32 or ry%32:
            raise ValueError('ROI must start on the 32 px grid and stay within the analysis image.')
        source_hash=digest(source)
        out=Path(output_dir);out.mkdir(parents=True,exist_ok=False)
        started=time.perf_counter();patches=[];pending=[];images=[];examined=0;skipped=0
        csv_path=out/'predictions.csv'
        fields=['id','row','col','x','y','width','height','p','tissue_fraction']
        config={'purpose':purpose,'target_mpp':target_mpp,'coordinate_scale':scale,'roi_analysis_pixels':roi,'max_candidates':max_candidates,'batch_size':batch_size,'min_tissue':min_tissue,'source_sha256':source_hash,'model_sha256':model_sha256,'preprocessing':'RGB uint8 to float32 / 255.0','context_size':96,'support_size':32,'stride':32,'sampling_order':'row-major within ROI; candidate cap includes skipped background'}
        (out/'run_config.json').write_text(json.dumps(config,indent=2))
        with csv_path.open('x',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()
            def flush():
                if not images:return
                batch=np.stack(images).astype(np.float32)/255.0
                predictions=np.asarray(predictor(batch))
                if predictions.shape not in ((len(images),),(len(images),1)):
                    raise ValueError('Expected one sigmoid probability per patch.')
                values=predictions.reshape(-1)
                if not np.all(np.isfinite(values)) or np.any(values<0) or np.any(values>1):
                    raise ValueError('Model output is not finite probability data.')
                for patch,value in zip(pending,values):
                    patch['p']=float(value);patches.append(patch);writer.writerow(patch)
                stream.flush();images.clear();pending.clear()
            try:
                for row in range(ry//32,math.ceil((ry+rh)/32)):
                    for col in range(rx//32,math.ceil((rx+rw)/32)):
                        if examined>=max_candidates:break
                        examined+=1;x,y=col*32,row*32
                        image=context(slide,x,y,scale);fraction=tissue_fraction(image)
                        if fraction<min_tissue:skipped+=1;continue
                        pending.append({'id':f'r{row}c{col}','row':row,'col':col,'x':x,'y':y,'width':min(32,aw-x),'height':min(32,ah-y),'tissue_fraction':round(fraction,4)})
                        images.append(image)
                        if len(images)>=batch_size:flush()
                    if examined>=max_candidates:break
                flush()
            except Exception:
                (out/'FAILED.txt').write_text('Run incomplete. predictions.csv contains only completed batches; do not treat it as a complete slide evaluation.')
                raise
        thumbnail=slide.get_thumbnail((1400,1400)).convert('RGB');buffer=io.BytesIO();thumbnail.save(buffer,'JPEG',quality=85)
        thumbnail_uri='data:image/jpeg;base64,'+base64.b64encode(buffer.getvalue()).decode()
        def mpp(key):
            try:
                value=float(slide.properties.get(key,''))
                return value if value>0 and math.isfinite(value) else None
            except ValueError:return None
        mx,my=mpp('openslide.mpp-x'),mpp('openslide.mpp-y')
        roi_cells=(math.ceil((rx+rw)/32)-rx//32)*(math.ceil((ry+rh)/32)-ry//32)
        coverage={'examined_candidates':examined,'scored_supports':len(patches),'background_skipped':skipped,'roi_candidates':roi_cells,'roi_scan_complete':examined==roi_cells,'whole_slide_scan_complete':roi==[0,0,aw,ah] and examined==roi_cells,'tissue_rule':'heuristic central-support brightness/chroma filter; not validated tissue segmentation'}
        sid=source_slide_id or 'sha256:'+source_hash
        result={'schema_version':1,'mode':'colab','purpose':purpose,'source_slide_id':sid,'source_sha256':source_hash,'coordinate_scale':scale,'slide':{'id':sid,'name':source.name,'width':aw,'height':ah,'mpp_x':mx*scale if mx else None,'mpp_y':my*scale if my else None},'viewer_slide':{'id':sid,'name':source.name,'width':width,'height':height,'levels':slide.level_count,'vendor':slide.properties.get('openslide.vendor','Unknown'),'mpp_x':mx,'mpp_y':my,'objective':None,'bytes':source.stat().st_size,'overview':thumbnail_uri},'grid':{'rows':rows,'cols':cols,'stride':32},'model':{'name':model_name,'sha256':model_sha256,'input_size':96,'support_size':32,'preprocessing':'RGB / 255.0'},'coverage':coverage,'patches':patches,'diagnosis_generated':False}
        payload=json.dumps(result,separators=(',',':'),allow_nan=False)
        if len(payload.encode())>16_000_000:
            raise ValueError('Result exceeds 16 MB browser limit. Use a smaller ROI or candidate cap; CSV remains saved.')
        (out/'dashboard-result.json').write_text(payload)
        manifest={**config,**coverage,'elapsed_seconds':round(time.perf_counter()-started,2),'utc_completed':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),'numpy':np.__version__,'openslide':openslide.__library_version__,'predictions_sha256':digest(csv_path),'overview_only_in_dashboard':True,'diagnosis_generated':False}
        (out/'manifest.json').write_text(json.dumps(manifest,indent=2))
        return out/'dashboard-result.json',manifest
    finally:
        slide.close()


In [ ]:
# 5. Load the saved model without optimizer state; verify its input and output.
MODEL_HASH = digest(MODEL_PATH)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
assert tuple(model.input_shape[1:]) == (96, 96, 3), f'Unexpected input: {model.input_shape}'
assert tuple(model.output_shape[1:]) == (1,), f'Unexpected output: {model.output_shape}'
def predict(batch):
    return model(batch, training=False).numpy()
probe = predict(np.zeros((1,96,96,3), dtype=np.float32))
assert probe.shape == (1,1) and np.isfinite(probe).all() and ((probe>=0)&(probe<=1)).all()
print('Model ready for inference; SHA-256:', MODEL_HASH)


In [ ]:
# 6. Process into a NEW folder. Existing pilot and Paper 1 files remain untouched.
import uuid
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = ROOT / 'Paper_2' / 'Colab_Runs' / (stamp + '_' + uuid.uuid4().hex[:8])
result_path, manifest = process_slide(
    SLIDE_PATH, predict, RUN_DIR,
    model_sha256=MODEL_HASH, model_name=MODEL_PATH.name,
    purpose=PURPOSE, target_mpp=TARGET_MPP,
    max_candidates=MAX_CANDIDATES, batch_size=BATCH_SIZE,
    min_tissue=MIN_TISSUE_FRACTION, roi=ROI)
assert digest(MODEL_PATH) == MODEL_HASH, 'Model file changed unexpectedly.'
manifest['tensorflow'] = tf.__version__
manifest['model_file_unchanged'] = True
(RUN_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
print('Dashboard result:', result_path)
if PURPOSE == 'technical_smoke_test':
    print('TECHNICAL TEST ONLY: public skin sample, not target-domain model evaluation. Nominal PCam input scale used.')


In [ ]:
# 7. Download the result for the browser dashboard.
from google.colab import files
files.download(str(result_path))
print('Open https://projectpatho.pages.dev → Import model results → select dashboard-result.json.')
print('CSV, configuration and manifest remain in your new Drive run folder.')
